In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime, timedelta

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainfallFSS"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Setup
Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"#; spinup_hours="-16"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSubsetting import DataSubsetting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#CALCULATING FUNCTIONS

In [ ]:
# #Loading Radar Mask #decided not to use here
# RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData_NSSL) 

In [ ]:
def SubsetData_Time(data,yearmonthday):
    data_T = data.sel(time=yearmonthday)
    # print(data_T.time) #testing
    return data_T

In [ ]:
def GetMean(variableSubset):
    #(1/A) times integral of phi dA 
    #dA is [(R*cos(Lat)dLon)][RdLat] = R^2 cos(Lat)dLatdLon ==> weight is simply cos(Lat)
    weights = np.cos(np.deg2rad(variableSubset.latitude))
    variableMean = variableSubset.weighted(weights).mean(
        dim=("latitude", "longitude"),
        skipna=True
    )
    return variableMean

In [ ]:
# #DATA LOADING FOR NCEP/EMC LEVEL IV DATA (*OLD*)

# def ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory, region='conus'):
#     yearmonth = yearmonthdayhour[0:6]
#     inputPath = os.path.join(inputDirectory,yearmonth,f"st4_{region}.{yearmonthdayhour}.01h.grb2")
#     precipData = xr.open_dataset(inputPath, engine="cfgrib")['tp']
#     #Note: data in kg/m^2 = mm since rho = m/V = m/A/h where m/A = 1 ==> h = 1e-3 m = 1 mm
#     return precipData

# def GetAccumulatedPrecipData_LevelIV(ModelData): 
#     #getting inputDirectory
#     inputDirectory = os.path.join(DirectoryManager.dataDirectory,
#                                   f"Observation_Data/{ModelData.region}/StageIV_PrecipData")
#     print(f"reading from {inputDirectory}")

#     #getting yearmonthdayhour list
#     dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
#     start = dt_list[0]
#     target = start + timedelta(hours=12)
    
#     # Find the entry closest to +12 hours
#     closest = min(dt_list, key=lambda x: abs(x - target))
    
#     closest_idx = dt_list.index(closest)
#     times = ModelData.timeStrings[closest_idx:]
#     yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
#                                 for t in times})
    
#     for count, yearmonthdayhour in tqdm(enumerate(yearmonthdayhours), total=len(yearmonthdayhours)):
#         if count == 0:
#             precipData_T = ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory)
#         else:
#             precipData_T += ReadPrecipData_LevelIV(yearmonthdayhour, inputDirectory)

#     precipData_T = precipData_T.assign_coords(longitude = precipData_T.longitude - 360)
#     return precipData_T

# precipData_T = GetAccumulatedPrecipData_LevelIV(ModelData_NSSL)
# precipData_T_Subset = DataSubsetting_Class.SubsetDataRegion_Curvilinear(precipData_T, ModelData_NSSL)

In [ ]:
def GetdtMinutes(t, t_prev):
    """
    Calculate the time difference in minutes between two datetimes.
    """
    dtMinutes = (t - t_prev).total_seconds() / 60.0
    return dtMinutes


def CalculateRainRate(data, data_prev, time, time_prev):
    """
    Calculate instantaneous rain rate between two timesteps.
    """
    dtMinutes = GetdtMinutes(time, time_prev)
    dtHours = dtMinutes / 60.0
    rainRate = (data - data_prev) / dtHours
    return rainRate


def GetRainRateData_Model(ModelData,t,t_prev):
    data = ModelData.GetDataTimestep(t,printout=False)
    data_prev = ModelData.GetDataTimestep(t_prev,printout=False)
    precipData_model_T = data['rainnc']+data['rainc']
    precipData_model_T_prev = data_prev['rainnc']+data_prev['rainc']


    # --- subset the region ---
    precipData_model_T_Subset = DataSubsetting_Class.SubsetDataRegion(
        precipData_model_T, ModelData
    )
    precipData_model_T_Subset_prev = DataSubsetting_Class.SubsetDataRegion(
        precipData_model_T_prev, ModelData
    )

    t_datetime = datetime.strptime(t, "%Y-%m-%d_%H.%M.%S")
    t_prev_datetime = datetime.strptime(t_prev, "%Y-%m-%d_%H.%M.%S")
    rainRate = CalculateRainRate(data=precipData_model_T_Subset,
                                 data_prev=precipData_model_T_Subset_prev,
                                 time=t_datetime,time_prev=t_prev_datetime)
    return rainRate


def GetRainRateData_Observations(ModelData,yearmonthdayhour,yearmonthdayhour_prev):

    inputDirectory = GetMRMS_QPE_DataDirectory(ModelData)
    print(f"reading from {inputDirectory}")
    
    data = ReadPrecipData_MRMS_QPE(ModelData,yearmonthdayhour,inputDirectory)
    data_prev = ReadPrecipData_MRMS_QPE(ModelData,yearmonthdayhour_prev,inputDirectory)

    # --- make a copy for coordinate fixes ---
    if data is None or data_prev is None:
        return None
    data = data.assign_coords(
        longitude = data.longitude - 360
    ).sortby("latitude")
    data_prev = data_prev.assign_coords(
        longitude = data_prev.longitude - 360
    ).sortby("latitude")

    # --- subset the region ---
    precipData_observations_T_Subset = DataSubsetting_Class.SubsetDataRegion(
        data, ModelData
    )
    precipData_observations_T_Subset_prev = DataSubsetting_Class.SubsetDataRegion(
        data_prev, ModelData
    )

    # --- interpolating observations to model domain ---
    precipData_observations_T_Subset = precipData_observations_T_Subset.interp(latitude=ModelData.latitude,
                                                                               longitude=ModelData.longitude,
                                                                               method="nearest")
    precipData_observations_T_Subset_prev = precipData_observations_T_Subset_prev.interp(latitude=ModelData.latitude,
                                                                               longitude=ModelData.longitude,
                                                                                         method="nearest")
    
    t_datetime = datetime.strptime(yearmonthdayhour, "%Y%m%d%H")
    t_prev_datetime = datetime.strptime(yearmonthdayhour_prev, "%Y%m%d%H")
    rainRate = CalculateRainRate(data=precipData_observations_T_Subset,
                                 data_prev=precipData_observations_T_Subset_prev,
                                 time=t_datetime,time_prev=t_prev_datetime)
    return rainRate

In [ ]:
#DATA LOADING FOR MRMS QPE DATA

def CorrectSimulationDates(ModelData, simulationDates):
    if int(ModelData.spinup_hours) <= 0:
        # Convert first date to Timestamp
        first_date = pd.to_datetime(simulationDates[0])
        # Subtract one day
        prev_date = (first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Prepend
        simulationDates = [prev_date] + simulationDates
    return simulationDates
    
def GetMRMS_QPE_DataDirectory(ModelData, product="MultiSensor_QPE_01H_Pass2_00.00"):
    simulationDates = ModelData.simulationDates
    simulationDates2 = CorrectSimulationDates(ModelData, simulationDates)

    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/{ModelData.region}/MRMS_RadarData",
             f"{simulationDates2[0]}_{simulationDates2[-1]}",product)
    return inputDirectory
    
def ReadPrecipData_MRMS_QPE(ModelData, yearmonthdayhour, inputDirectory):
    yearmonthday = yearmonthdayhour[0:8]
    hour = yearmonthdayhour[8:]

    MRMS_region = "CONUS" if ModelData.region=="TRACER" else "HAWAII"
    inputPath = os.path.join(inputDirectory,f"MRMSQPE_{MRMS_region}_{ModelData.region}_{yearmonthday}-{hour}0000.nc")
    try:
        precipData = xr.open_dataset(inputPath)["MultiSensor_QPE_01H_Pass2_00.00"].isel(time=0)
    except:
        print(f"{inputPath} does not exist in MRMS data ==> skipping")
        precipData = None
    #Note: data is in mm units
    return precipData

In [ ]:
def PrepareTimes(ModelData):
    """Return aligned times and previous times for looping."""
    
    #getting yearmonthdayhour list
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
    start = dt_list[0]
    target = start #+ timedelta(hours=13) #only using for rainfall histogram
    # print(f"start_time: {target}")
    
    # Find the entry closest to +12 hours
    closest = min(dt_list, key=lambda x: abs(x - target))
    
    closest_idx = dt_list.index(closest)
    times = ModelData.timeStrings[closest_idx:]
    times = [t for t in times if datetime.strptime(t, "%Y-%m-%d_%H.%M.%S").minute == 0]
    times_prev = [None] + times[:-1]
    times_prev = [t if t is None else t for t in times_prev
                  if t is None or datetime.strptime(t, "%Y-%m-%d_%H.%M.%S").minute == 0]

    yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
                                for t in times})
    yearmonthdayhours_prev = [None if t is None else t.replace('-', '').replace('_', '').replace('.', '')[:10]
                              for t in times_prev]
    
    return times, times_prev, yearmonthdayhours, yearmonthdayhours_prev

In [ ]:
## CalculateFSS_scores

# Leeuwenburg, T., Loveday, N., Ebert, E. E., Cook, H., Khanarmuei, M., Taggart, R. J., Ramanathan, N., Carroll, M., Chong, S., Griffiths, A., & Sharples, J. (2024). 
# scores: A Python package for verifying and evaluating models and predictions with xarray. Journal of Open Source Software, 9(99), 6889. https://doi.org/10.21105/joss.06889
# https://scores.readthedocs.io/en/2.0.0/tutorials/Fractions_Skill_Score.html

# pip install scores
from scores.spatial import fss_2d_single_field
from scores.fast.fss.typing import FssComputeMethod

def CalculateFSS_scores(forecast,observation,threshold,window_size):
    compute_method = FssComputeMethod.NUMPY
    threshold_operator = np.greater_equal
    # threshold_operator = np.greater
    
    fs_score = fss_2d_single_field(
        forecast,
        observation,
        event_threshold=threshold,
        window_size=window_size,           # same interpretation as 'scale'
        threshold_operator=threshold_operator,
        compute_method=compute_method # default and fastest
    )
    return fs_score*100

In [710]:
def GetPrecipData_FSS(ModelData): 

    [times, times_prev, yearmonthdayhours, yearmonthdayhours_prev] = PrepareTimes(ModelData)
    
    FSSList = []; timeList = []
    for count, (t_prev, t, yearmonthdayhour_prev, yearmonthdayhour) in tqdm(
            enumerate(zip(times_prev, times, yearmonthdayhours_prev, yearmonthdayhours)),
            total=len(times)
        ):
        if t_prev is None:
            continue

        rainRate_model = GetRainRateData_Model(ModelData,t,t_prev) #Getting Model Rain Rate
        if rainRate_model is None:
            FSSList.append(np.nan)
            timeList.append(t)
            continue
        rainRate_model = rainRate_model.where(rainRate_model > 0)
        rainRate_observations = GetRainRateData_Observations(ModelData,yearmonthdayhour,yearmonthdayhour_prev)
        if rainRate_observations is None:
            FSSList.append(np.nan)
            timeList.append(t)
            continue
        rainRate_observations = rainRate_observations.where(rainRate_observations > 0)
    
        FSS = CalculateFSS_scores(forecast=rainRate_model.data,
                                  observation=rainRate_observations.data,
                                  threshold=0,
                                  window_size=(10,10))
    
        FSSList.append(FSS)
        timeList.append(t)

    timeList = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in timeList]
    return np.array(FSSList),np.array(timeList)

In [ ]:
####################################
#CALCULATING

In [ ]:
[FSSList_NSSL,timeList] = GetPrecipData_FSS(ModelData_NSSL)
[FSSList_TEMPO,_] = GetPrecipData_FSS(ModelData_TEMPO)

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
#HELPER FUNCTIONS
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass


    time_before = times[0] - timedelta(hours=1)
    times = np.insert(times, 0, time_before)
    ax.set_xlim(times.min(), times.max())

In [ ]:
def MakePlot(timeList,FSSList_NSSL,FSSList_TEMPO):    
    
    fig, ax = plt.subplots(figsize=(10, 5))  # create figure and axes
    
    # Plot on the axes
    ax.plot(timeList, FSSList_NSSL, color='blue', label='NSSL')
    ax.plot(timeList, FSSList_TEMPO, color='green', label='TEMPO')
    
    # Rotate x-axis labels
    ax.tick_params(axis='x', rotation=45)
    
    # Labels
    ax.set_xlabel('Time')
    ax.set_ylabel('FSS (%)')
    
    # Title
    ax.set_title(f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
                 fontweight='bold',
                 fontsize=14)
    
    # Legend
    ax.legend()
    
    # Grid
    ax.grid(True)
    
    SetXLimitsDatetime(ax, timeList)
    fig.tight_layout()
    
    return fig

In [ ]:
def SaveFigure(fig, ModelData1, ModelData2, dpi=150):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"RainfallFSS.png"
    )

    # # --- Save figure ---
    # FigurePlotting_Class.SaveUniformFigure(fig, outputFilePath)
    fig.savefig(outputFilePath, dpi=dpi, bbox_inches="tight",pad_inches=0.02)
    plt.close(fig)
    print(f"Saved image: {outputFilePath}")

In [ ]:
####################################
#PLOTTING

In [ ]:
fig = MakePlot(timeList,FSSList_NSSL,FSSList_TEMPO)
SaveFigure(fig, ModelData_NSSL, ModelData_TEMPO)